In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
import re
import nltk

In [2]:
nltk_resources = [
    'tokenizers/punkt', 
    'averaged_perceptron_tagger_eng',
    'corpora/stopwords', 
    'help/tagsets'
]

for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except IndexError:
        nltk.download(resource)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [3]:
data_home = "../input"
output_dir = "../working"

source_file_dir = '/kaggle/input/datasets/isabellouisedelgado/classical-gothic-novels'
source_file_dir

'/kaggle/input/datasets/isabellouisedelgado/classical-gothic-novels'

In [4]:
# This structure matches all the texts
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

# Gutenberg boilerplate clip patterns (common to all texts)
clip_pats_gutenberg = [
    r"\*\*\*\s*START OF",
    r"\*\*\*\s*END OF"
]

# Roman numeral character class for reuse
roman = '[IVXLCM]+'

chap_pats = chap_pats = [
    (53685, rf"^(?:CHAPTER\s+{roman}\.|Tale of the Spaniard\.)\s*$"),
    (41445, rf"^(?:LETTER|CHAPTER)\s+{roman}\.$"),
    (6087,  rf"^\s*THE VAMPYRE\.\s*$"),
]

In [5]:
source_file_list = sorted(glob(f"{source_file_dir}/*.*"))
source_file_list

['/kaggle/input/datasets/isabellouisedelgado/classical-gothic-novels/MATURIN_CHARLES_MELMOTH_THE_WANDERER-pg53685.txt',
 '/kaggle/input/datasets/isabellouisedelgado/classical-gothic-novels/POLIDORI_JOHN_THE_VAMPYRE-pg6087.txt',
 '/kaggle/input/datasets/isabellouisedelgado/classical-gothic-novels/SHELLEY_MARY_FRANKENSTEIN-pg41445.txt']

In [6]:
book_data = []
for source_file_path in source_file_list:
    book_id = int(source_file_path.split('-')[-1].split('.')[0].replace('pg',''))
    raw_title = source_file_path.split('/')[-1].split('-')[0].replace('_', ' ')
    book_data.append((book_id, source_file_path, raw_title))

book_data

[(53685,
  '/kaggle/input/datasets/isabellouisedelgado/classical-gothic-novels/MATURIN_CHARLES_MELMOTH_THE_WANDERER-pg53685.txt',
  'MATURIN CHARLES MELMOTH THE WANDERER'),
 (6087,
  '/kaggle/input/datasets/isabellouisedelgado/classical-gothic-novels/POLIDORI_JOHN_THE_VAMPYRE-pg6087.txt',
  'POLIDORI JOHN THE VAMPYRE'),
 (41445,
  '/kaggle/input/datasets/isabellouisedelgado/classical-gothic-novels/SHELLEY_MARY_FRANKENSTEIN-pg41445.txt',
  'SHELLEY MARY FRANKENSTEIN')]

In [7]:
LIB = pd.DataFrame(book_data, columns=['book_id','source_file_path', 'raw_title'])\
    .set_index('book_id').sort_index()
LIB

,source_file_path,raw_title
book_id,,
6087,/kaggle/input/datasets/isabellouisedelgado/cla...,POLIDORI JOHN THE VAMPYRE
41445,/kaggle/input/datasets/isabellouisedelgado/cla...,SHELLEY MARY FRANKENSTEIN
53685,/kaggle/input/datasets/isabellouisedelgado/cla...,MATURIN CHARLES MELMOTH THE WANDERER


In [8]:
try:
    LIB['author'] = LIB.raw_title.apply(lambda x: ', '.join(x.split()[:2]))
    LIB['title'] = LIB.raw_title.apply(lambda x: ' '.join(x.split()[2:]))
    LIB = LIB.drop('raw_title', axis=1)

# Skip if this cell has already been run
except AttributeError:
    pass

LIB

,source_file_path,author,title
book_id,,,
6087,/kaggle/input/datasets/isabellouisedelgado/cla...,"POLIDORI, JOHN",THE VAMPYRE
41445,/kaggle/input/datasets/isabellouisedelgado/cla...,"SHELLEY, MARY",FRANKENSTEIN
53685,/kaggle/input/datasets/isabellouisedelgado/cla...,"MATURIN, CHARLES",MELMOTH THE WANDERER


In [9]:
LIB['chap_pat'] = pd.Series({pat[0]:pat[1] for pat in chap_pats})
LIB

,source_file_path,author,title,chap_pat
book_id,,,,
6087,/kaggle/input/datasets/isabellouisedelgado/cla...,"POLIDORI, JOHN",THE VAMPYRE,^\s*THE VAMPYRE\.\s*$
41445,/kaggle/input/datasets/isabellouisedelgado/cla...,"SHELLEY, MARY",FRANKENSTEIN,^(?:LETTER|CHAPTER)\s+[IVXLCM]+\.$
53685,/kaggle/input/datasets/isabellouisedelgado/cla...,"MATURIN, CHARLES",MELMOTH THE WANDERER,^(?:CHAPTER\s+[IVXLCM]+\.|Tale of the Spaniard...


In [10]:
# Add clip_pats and chap_pat to LIB
LIB['clip_pats'] = None
LIB['chap_pat'] = None

for i in LIB.index:
    LIB.at[i, 'clip_pats'] = clip_pats_gutenberg

for book_id, pat in chap_pats:
    LIB.at[book_id, 'chap_pat'] = pat

# Verify it looks right before running corpus
LIB[['title', 'clip_pats', 'chap_pat']]

,title,clip_pats,chap_pat
book_id,,,
6087,THE VAMPYRE,"[\*\*\*\s*START OF, \*\*\*\s*END OF]",^\s*THE VAMPYRE\.\s*$
41445,FRANKENSTEIN,"[\*\*\*\s*START OF, \*\*\*\s*END OF]",^(?:LETTER|CHAPTER)\s+[IVXLCM]+\.$
53685,MELMOTH THE WANDERER,"[\*\*\*\s*START OF, \*\*\*\s*END OF]",^(?:CHAPTER\s+[IVXLCM]+\.|Tale of the Spaniard...


In [11]:
def tokenize_source(book_id):
    src_file = LIB.loc[book_id].source_file_path
    
    # Convert the raw text into a dataframe
    text_lines = open(src_file, 'r', encoding="utf-8-sig").readlines()
    LINES = pd.DataFrame({'book_id': book_id, 'line_str': text_lines})
    LINES.line_str = LINES.line_str.str.strip()
    
    # Clip the cruft

    clip_pats = LIB.loc[book_id, 'clip_pats']
    start_pat = clip_pats[0]
    end_pat = clip_pats[1]
    start = LINES.line_str.str.contains(start_pat, regex=True)
    end = LINES.line_str.str.contains(end_pat, regex=True)
    try:
        start_line_num = LINES.loc[start].index[0]
    except IndexError:
        raise ValueError("Clip start pattern not found.")
    try:
        end_line_num = LINES.loc[end].index[0]
    except IndexError:
        raise ValueError("Clip end pattern not found.")
    LINES = LINES.loc[start_line_num + 1 : end_line_num - 2].copy()
    
    # Chunk chapters
    chap_pat = LIB.loc[book_id, 'chap_pat']
    chap_lines = LINES.line_str.str.contains(chap_pat, regex=True, case=True)
    LINES.loc[chap_lines, 'chap_num'] = [i+1 for i in range(LINES.loc[chap_lines].shape[0])]
    LINES.chap_num = LINES.chap_num.ffill()
    LINES = LINES.loc[~LINES.chap_num.isna()]
    LINES = LINES.loc[~chap_lines]
    LINES.chap_num = LINES.chap_num.astype('int')
    CHAPS = LINES.groupby('chap_num', group_keys=True)['line_str']\
        .apply(lambda x: '\n'.join(x)).to_frame('chap_str')
    del LINES
    
    # Build OHCO records with plain Python loops
    records = []
    for chap_num, chap_str in CHAPS.chap_str.items():
        for para_num, para_str in enumerate(chap_str.split('\n\n')):
            if not para_str.strip():
                continue
            for sent_num, sent_str in enumerate(nltk.sent_tokenize(para_str)):
                for token_num, (token_str, pos) in enumerate(nltk.pos_tag(nltk.word_tokenize(sent_str))):
                    records.append((chap_num, para_num, sent_num, token_num, token_str, pos))
    del CHAPS
    
    # Build TOKENS dataframe
    TOKENS = pd.DataFrame(records, columns=['chap_num', 'para_num', 'sent_num', 'token_num', 'token_str', 'pos'])
    TOKENS['pos_group'] = TOKENS.pos.str[:2]
    TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"[\W_]+", "", regex=True)
    TOKENS = TOKENS[TOKENS.term_str != ''].copy()
    TOKENS = TOKENS.set_index(['chap_num', 'para_num', 'sent_num', 'token_num'])
    
    return TOKENS

In [12]:
corpus = []
for i in LIB.index:
    print(LIB.loc[i].title)
    corpus.append(tokenize_source(i))
print("Done")
CORPUS = pd.concat(corpus, keys=LIB.index)
CORPUS.index.names = OHCO
del corpus

THE VAMPYRE
FRANKENSTEIN
MELMOTH THE WANDERER
Done


In [13]:
CORPUS

token_str  pos pos_group  \
book_id chap_num para_num sent_num token_num                                
6087    1        1        0        0          INTRODUCTION  NNP        NN   
                 2        0        0                   THE   DT        DT   
                                   1          superstition   NN        NN   
                                   2                  upon   IN        IN   
                                   3                 which  WDT        WD   
...                                                    ...  ...       ...   
53685   6        85       1        12                   of   IN        IN   
                                   13                   it  PRP        PR   
                          2        0                     I  PRP        PR   
                                   1                  gave  VBD        VB   
                                   2                   the   DT        DT   

                                                  term_str  
book_id chap_num para_num sent_num token_num                
6087    1        1        0        0          introduction  
                 2        0        0                   the  
                                   1          superstition  
                                   2                  upon  
                                   3                 which  
...                                                    ...  
53685   6        85       1        12                   of  
                                   13                   it  
                          2        0                     i  
                                   1                  gave  
                                   2                   the  

[140082 rows x 4 columns]

In [14]:
LIB.to_csv('/kaggle/working/LIB_classical_gothic.csv')
CORPUS.to_csv('/kaggle/working/CORPUS_classical_gothic.csv')